# tieuluan3 · Knowledge Graph và RAG

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [ ]:
PROJECT_NAME = 'tieuluanfinal'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
import assignment_runtime as _final_runtime
if PROJECT_NAME not in _final_runtime.PROJECTS:
    _final_runtime.PROJECTS = (*_final_runtime.PROJECTS, PROJECT_NAME)
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


### `apps.api.knowledge_rag_service`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
"""Lazy, hash-verified KG-RAG runtime for the Phase 16 API."""

from __future__ import annotations

import gzip

import hashlib

import json

import threading

import time

from dataclasses import dataclass

from pathlib import Path

from typing import Any, Callable, Mapping

import joblib

from term_paper.tools.phase13_graph import normalize_label

from term_paper.tools.phase14_rag import grounded_answer

from term_paper.tools.phase15_kgrag import graph_filter_ranking, weighted_rrf

ALLOWED_DOMAINS = {"auto", "medical", "finance"}

ALLOWED_TOOLS = {"medical": "medical_retriever", "finance": "finance_retriever"}

MAX_QUESTION_CHARACTERS = 500

MEDICAL_TERMS = {
    "diabetes", "diabetic", "insulin", "glucose", "blood sugar", "a1c",
    "kidney", "neuropathy", "retinopathy", "hypoglycemia", "hyperglycemia",
}

FINANCE_TERMS = {
    "revenue", "expense", "profit", "income", "asset", "liability", "cash flow",
    "margin", "fiscal", "annual report", "acquisition", "lease", "tax", "equity",
    "shares", "ebitda", "debt", "cost", "percentage change",
}

PROMPT_INJECTION_PATTERNS = {
    "ignore previous", "ignore all instructions", "system prompt", "developer message",
    "reveal prompt", "show hidden", "execute code", "run command", "shell command",
    "rm rf", "override safety", "bypass guardrail",
}

MEDICAL_SAFETY_PATTERNS = {
    "personalized insulin dose", "insulin dose", "dosage", "diagnose me", "diagnose my",
    "prescribe", "stop prescribed", "stop my medicine", "medical emergency",
}

FINANCIAL_SAFETY_PATTERNS = {
    "best stock to buy", "should i buy", "should i sell", "recommend a stock",
    "investment advice", "stock price tomorrow", "guaranteed return", "buy this week",
}



#### AgentInputError

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
class AgentInputError(ValueError):
    """Safe validation error for an Agent request."""



#### ArtifactUnavailableError

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
class ArtifactUnavailableError(RuntimeError):
    """Packaged runtime is absent or fails integrity checks."""



#### AgentTimeoutError

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
class AgentTimeoutError(TimeoutError):
    """Local inference exceeded its configured deadline."""



#### _sha256

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



#### validate_question

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
def validate_question(value: Any) -> str:
    if not isinstance(value, str):
        raise AgentInputError("question phải là chuỗi văn bản.")
    question = " ".join(value.split())
    if len(question) < 3:
        raise AgentInputError("question phải có ít nhất 3 ký tự.")
    if len(question) > MAX_QUESTION_CHARACTERS:
        raise AgentInputError(f"question không được vượt quá {MAX_QUESTION_CHARACTERS} ký tự.")
    return question



#### route_domain

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
def route_domain(question: str, requested_domain: str = "auto") -> str | None:
    domain = str(requested_domain or "auto").casefold().strip()
    if domain not in ALLOWED_DOMAINS:
        raise AgentInputError("domain chỉ nhận auto, medical hoặc finance.")
    if domain != "auto":
        return domain
    normalized = normalize_label(question)
    medical_score = sum(term in normalized for term in MEDICAL_TERMS)
    finance_score = sum(term in normalized for term in FINANCE_TERMS)
    if medical_score == finance_score:
        return None
    return "medical" if medical_score > finance_score else "finance"



#### guardrail_reason

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
def guardrail_reason(question: str, domain: str | None) -> str | None:
    normalized = normalize_label(question)
    if any(pattern in normalized for pattern in PROMPT_INJECTION_PATTERNS):
        return "prompt_injection"
    if domain == "medical" and any(pattern in normalized for pattern in MEDICAL_SAFETY_PATTERNS):
        return "medical_safety"
    if domain == "finance" and any(pattern in normalized for pattern in FINANCIAL_SAFETY_PATTERNS):
        return "financial_safety"
    return None



#### RuntimeBundle

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
@dataclass
class RuntimeBundle:
    manifest: dict[str, Any]
    chunks: dict[str, dict[str, dict[str, Any]]]
    indexes: dict[str, dict[str, Any]]
    graph: dict[str, dict[str, dict[str, Any]]]



#### KnowledgeRagService

In [ ]:
%%inline_module apps.api.knowledge_rag_service apps/api/knowledge_rag_service.py both
class KnowledgeRagService:
    """Thread-safe lazy loader and deterministic two-tool Agent router."""

    def __init__(
        self,
        workspace: Path,
        runtime_dir: Path | None = None,
        loader: Callable[[], RuntimeBundle] | None = None,
        timeout_ms: int = 8_000,
    ):
        self.workspace = Path(workspace)
        self.runtime_dir = runtime_dir or self.workspace / "apps/api/models"
        self.manifest_path = self.runtime_dir / "kg_manifest.json"
        self.timeout_ms = int(timeout_ms)
        self._custom_loader = loader
        self._bundle: RuntimeBundle | None = None
        self._lock = threading.Lock()
        self._integrity: dict[str, Any] | None = None

    def status(self) -> dict[str, Any]:
        version = None
        if self.manifest_path.exists():
            try:
                version = json.loads(self.manifest_path.read_text(encoding="utf-8")).get("artifact_version")
            except (OSError, json.JSONDecodeError):
                pass
        state = "ready" if self._bundle is not None else ("cold" if self.manifest_path.exists() else "unavailable")
        return {
            "status": state,
            "artifact_version": version,
            "loaded": self._bundle is not None,
            "allowed_tools": sorted(ALLOWED_TOOLS.values()),
        }

    def readiness(self) -> dict[str, Any]:
        try:
            return {"status": "ready", **self._verify_artifacts()}
        except (ArtifactUnavailableError, OSError, ValueError, json.JSONDecodeError):
            return {
                "status": "unavailable",
                "artifact_version": None,
                "hash_verified": False,
                "message": "KG-RAG runtime artifact is unavailable or invalid.",
            }

    def _verify_artifacts(self) -> dict[str, Any]:
        if self._integrity is not None:
            return dict(self._integrity)
        if not self.manifest_path.exists():
            raise ArtifactUnavailableError("runtime manifest missing")
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        files = manifest.get("files")
        if not isinstance(files, list) or not files:
            raise ArtifactUnavailableError("runtime manifest has no files")
        total = 0
        for item in files:
            path = (self.runtime_dir / str(item.get("path", ""))).resolve()
            if self.runtime_dir.resolve() not in path.parents:
                raise ArtifactUnavailableError("runtime artifact escaped package directory")
            if not path.is_file() or _sha256(path) != item.get("sha256"):
                raise ArtifactUnavailableError("runtime artifact hash mismatch")
            if path.stat().st_size != item.get("size_bytes"):
                raise ArtifactUnavailableError("runtime artifact size mismatch")
            total += path.stat().st_size
        if total > int(manifest.get("artifact_budget_bytes", 0)):
            raise ArtifactUnavailableError("runtime package exceeds artifact budget")
        self._integrity = {
            "artifact_version": manifest.get("artifact_version"),
            "hash_verified": True,
            "package_bytes": total,
            "budget_bytes": manifest.get("artifact_budget_bytes"),
        }
        return dict(self._integrity)

    def _load_bundle(self) -> RuntimeBundle:
        self._verify_artifacts()
        manifest = json.loads(self.manifest_path.read_text(encoding="utf-8"))
        chunks: dict[str, dict[str, dict[str, Any]]] = {"medical": {}, "finance": {}}
        with gzip.open(self.runtime_dir / manifest["chunks_file"], "rt", encoding="utf-8") as handle:
            for line in handle:
                if line.strip():
                    row = json.loads(line)
                    chunks[row["domain"]][row["chunk_id"]] = row
        with gzip.open(self.runtime_dir / manifest["graph_file"], "rt", encoding="utf-8") as handle:
            graph = json.load(handle)
        indexes = {
            domain: {
                kind: joblib.load(self.runtime_dir / relative)
                for kind, relative in manifest["indexes"][domain].items()
            }
            for domain in ("medical", "finance")
        }
        return RuntimeBundle(manifest=manifest, chunks=chunks, indexes=indexes, graph=graph)

    def _get_bundle(self) -> RuntimeBundle:
        if self._bundle is None:
            with self._lock:
                if self._bundle is None:
                    self._bundle = self._custom_loader() if self._custom_loader else self._load_bundle()
        return self._bundle

    @staticmethod
    def _graph_candidates(
        question: str, graph: Mapping[str, Mapping[str, Any]]
    ) -> tuple[set[str], list[str]]:
        normalized = normalize_label(question)
        query_terms = set(normalized.split())
        matched: list[tuple[str, Mapping[str, Any]]] = []
        for key, item in graph.items():
            key_terms = set(key.split())
            if key and (key in normalized or (len(key_terms) > 1 and key_terms <= query_terms)):
                matched.append((key, item))
        matched.sort(key=lambda pair: (-len(pair[0]), pair[0]))
        candidates: set[str] = set()
        path: list[str] = []
        for key, item in matched[:4]:
            candidates.update(str(value) for value in item.get("chunk_ids", []))
            if not path:
                path = ["user_question", str(item.get("label", key)), str(item.get("node_id", "entity"))]
        return candidates, path

    def _abstention(
        self, requested_domain: str, domain: str | None, reason: str, started: float
    ) -> dict[str, Any]:
        tool = ALLOWED_TOOLS.get(domain) if domain else None
        warning = (
            "Thông tin học thuật, không thay thế chẩn đoán hoặc tư vấn y khoa."
            if domain == "medical"
            else "Thông tin học thuật, không phải khuyến nghị đầu tư."
            if domain == "finance"
            else "Câu hỏi nằm ngoài hai miền dữ liệu medical và finance."
        )
        return {
            "answer": "",
            "domain": domain,
            "citations": [],
            "evidence": [],
            "route": {"requested_domain": requested_domain, "selected_domain": domain, "tool": tool},
            "graph_path": [],
            "abstained": True,
            "abstention_reason": reason,
            "latency_ms": round((time.perf_counter() - started) * 1000.0, 3),
            "artifact_version": self.status().get("artifact_version"),
            "warning": warning,
        }

    def query(self, question: Any, domain: str = "auto") -> dict[str, Any]:
        started = time.perf_counter()
        clean_question = validate_question(question)
        requested = str(domain or "auto").casefold().strip()
        selected = route_domain(clean_question, requested)
        reason = guardrail_reason(clean_question, selected)
        if reason:
            return self._abstention(requested, selected, reason, started)
        if selected is None:
            return self._abstention(requested, None, "out_of_domain", started)

        bundle = self._get_bundle()
        if (time.perf_counter() - started) * 1000.0 > self.timeout_ms:
            raise AgentTimeoutError("KG-RAG request exceeded its deadline")
        config = bundle.manifest["retrieval"]
        sparse = bundle.indexes[selected]["sparse"].search(clean_question, top_k=config["candidate_depth"])
        dense = bundle.indexes[selected]["dense"].search(clean_question, top_k=config["candidate_depth"])
        candidates, graph_path = self._graph_candidates(clean_question, bundle.graph[selected])
        graph_ranking = graph_filter_ranking(sparse, candidates, config["candidate_depth"])
        ranking = weighted_rrf(
            {"sparse": sparse, "dense": dense, "graph": graph_ranking},
            config["weights"], config["top_k"], config["rrf_k"],
        )
        chunk_lookup = bundle.chunks[selected]
        answer_config = bundle.manifest["answer_layer"]
        answer = grounded_answer(
            clean_question,
            ranking,
            {chunk_id: chunk_lookup[chunk_id]["text"] for chunk_id, _ in ranking if chunk_id in chunk_lookup},
            answer_config["minimum_overlap_terms"], answer_config["minimum_overlap_ratio"],
            answer_config["unsafe_patterns"], answer_config["maximum_characters"],
        )
        scores = {chunk_id: float(score) for chunk_id, score in ranking}
        evidence = [
            {
                "chunk_id": chunk_id,
                "document_id": chunk_lookup[chunk_id]["document_id"],
                "text": chunk_lookup[chunk_id]["text"],
                "score": round(scores[chunk_id], 8),
            }
            for chunk_id, _ in ranking[:5]
            if chunk_id in chunk_lookup
        ]
        citations = [
            {"chunk_id": chunk_id, "document_id": chunk_lookup[chunk_id]["document_id"]}
            for chunk_id in answer["citations"]
            if chunk_id in chunk_lookup
        ]
        if graph_path and ranking:
            graph_path = [*graph_path, ranking[0][0]]
        elapsed_ms = (time.perf_counter() - started) * 1000.0
        if elapsed_ms > self.timeout_ms:
            raise AgentTimeoutError("KG-RAG request exceeded its deadline")
        return {
            "answer": answer["answer"],
            "domain": selected,
            "citations": citations,
            "evidence": evidence,
            "route": {"requested_domain": requested, "selected_domain": selected, "tool": ALLOWED_TOOLS[selected]},
            "graph_path": graph_path,
            "abstained": bool(answer["abstained"]),
            "abstention_reason": answer["reason"] if answer["abstained"] else None,
            "latency_ms": round(elapsed_ms, 3),
            "artifact_version": bundle.manifest["artifact_version"],
            "warning": (
                "Thông tin học thuật, không thay thế chẩn đoán hoặc tư vấn y khoa."
                if selected == "medical"
                else "Thông tin học thuật, không phải khuyến nghị đầu tư."
            ),
        }



### `src.common.contracts`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
"""Offline Phase 22 specification guards, not a training or TEST executor.

jsonschema/referencing are already pinned by the shared uv lock. No network fetch
is registered: unknown external schema URIs fail closed.
"""

from __future__ import annotations

import csv

import hashlib

import json

import math

from datetime import datetime

from functools import lru_cache

from itertools import pairwise

from pathlib import Path

from jsonschema import Draft202012Validator, FormatChecker

from referencing import Registry, Resource

ROOT = Path(__file__).resolve().parents[2]

SCHEMAS = ROOT / "src/common/schemas/v3"

PROTOCOL = ROOT / "experiments/configs/v3/protocol_v3.json"

REQUIREMENT_IDS = {f"REQ-{number:02}" for number in range(1, 55)}



#### ContractError

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
class ContractError(ValueError):
    """A structural or semantic specification violation."""



#### read_json

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8-sig"))



#### canonical_sha256

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def canonical_sha256(value) -> str:
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()



#### load_schemas

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
@lru_cache(maxsize=1)
def load_schemas() -> dict:
    schemas = {
        path.name.removesuffix(".schema.json"): read_json(path) for path in SCHEMAS.glob("*.schema.json")
    }
    for schema in schemas.values():
        Draft202012Validator.check_schema(schema)
    return schemas



#### _finite

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _finite(value):
    if isinstance(value, float) and not math.isfinite(value):
        raise ContractError("All numeric values must be finite; undefined metrics use null with a reason")
    if isinstance(value, dict):
        for child in value.values():
            _finite(child)
    elif isinstance(value, list):
        for child in value:
            _finite(child)



#### _backend

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _backend(document):
    allowed = {"scratch": {"numpy"}, "keras": {"tensorflow", "torch"}, "pytorch": {"torch"}}
    if document["backend"] not in allowed[document["implementation"]]:
        raise ContractError("implementation/backend mismatch")



#### _dataset_family

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset_family(document):
    protocol = read_json(PROTOCOL)
    dataset = next((row for row in protocol["datasets"] if row["dataset_id"] == document["dataset_id"]), None)
    if dataset is None or dataset["chapter"] != protocol["families"][document["family"]]["chapter"]:
        raise ContractError("Model family/dataset mismatch")



#### _time

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _time(value):
    return datetime.fromisoformat(value)



#### _model

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _model(document):
    _backend(document)
    _dataset_family(document)
    policy = read_json(PROTOCOL)
    layers = document["layers"]
    eligible = [
        layer
        for layer in layers
        if layer["kind"] in policy["layer_policy"]["counted_kinds"] and layer["parameter_count"] > 0
    ]
    expected = policy["families"][document["family"]]
    if len(eligible) < expected["minimum_learnable_layers"]:
        raise ContractError("Too few learnable layers; pooling/normalization/time steps do not count")
    if [layer["kind"] for layer in eligible] != expected["blocks"]:
        raise ContractError("Learnable layer composition differs from the locked reference architecture")
    if sum(layer["parameter_count"] for layer in layers) != document["parameter_count"]:
        raise ContractError("Total parameter count does not match layers")
    functions = {function["id"]: function for function in document["functions"]}
    if len(functions) != len(document["functions"]) or len({layer["id"] for layer in layers}) != len(layers):
        raise ContractError("Duplicate function/layer IDs")
    if any(name not in functions for name in document["composition"]):
        raise ContractError("Composition references a missing function")
    for layer in layers:
        if layer["function_id"] not in functions:
            raise ContractError("Layer references a missing function")
    if document["family"] in {"rnn", "lstm"} and not document["recurrent_unroll"]:
        raise ContractError("Recurrent model needs a time/layer unroll explanation")



#### _split

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _split(document):
    partitions = document["partitions"]
    if {part["name"] for part in partitions} != {"train", "validation", "test"}:
        raise ContractError("Each partition must occur exactly once")
    if document["method"] == "temporal":
        for part in partitions:
            if (
                not part["target_start"]
                or not part["target_end"]
                or _time(part["target_start"]) > _time(part["target_end"])
            ):
                raise ContractError("Temporal partition needs a valid target time range")
        by_name = {part["name"]: part for part in partitions}
        times = [
            (_time(by_name[name]["target_start"]), _time(by_name[name]["target_end"]))
            for name in ("train", "validation", "test")
        ]
        if any(left[1] >= right[0] for left, right in pairwise(times)):
            raise ContractError("Temporal target time ranges overlap")



#### _run

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _run(document):
    _backend(document)
    _dataset_family(document)
    if document["track"] == "matched_subset" and not document["subset_sha256"]:
        raise ContractError("Matched-subset run needs its selected train ID hash")
    if document["split"] == "test" and not document["test_seal_id"]:
        raise ContractError("TEST metrics require a test seal ID")
    if _time(document["started_at"]) > _time(document["finished_at"]):
        raise ContractError("Run finish precedes start")
    efficiency = document["efficiency"]
    if efficiency["inference_p95_ms"] < efficiency["inference_p50_ms"]:
        raise ContractError("Inference p95 cannot be below p50")
    if document["device"] == "cpu" and efficiency["peak_vram_bytes"] != 0:
        raise ContractError("CPU run cannot claim GPU allocator memory")
    for metric in document["metrics"]:
        if (metric["value"] is None) != bool(metric["undefined_reason"]):
            raise ContractError("Undefined metric must use null and an explicit reason")



#### _dataset

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset(document):
    if document["status"] == "ready":
        if document["source"]["license_status"] != "approved":
            raise ContractError("Ready dataset requires approved license/access")
        if document["logical_size"]["records"] is None:
            raise ContractError("Ready dataset requires measured logical size")
        if document["kind"] != "task_view" and not document["files"]:
            raise ContractError("Ready dataset needs file hashes")
    if document["kind"] == "task_view" and not document["parent_dataset_ids"]:
        raise ContractError("Task view must reference source datasets")



#### _seal

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _seal(document):
    expected = 1 if document["state"] in {"evaluating", "evaluated", "failed_sealed"} else 0
    if document["evaluation_count"] != expected:
        raise ContractError("Seal state/evaluation count mismatch")
    if len(set(document["approved_run_ids"])) != len(document["approved_run_ids"]):
        raise ContractError("Seal approved run IDs must be unique")



#### _artifact

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _artifact(document):
    loaders = {
        "npz": "numpy_no_pickle",
        "keras": "keras_safe_mode",
        "torch_state_dict": "torch_weights_only",
        "legacy_trusted_joblib": "legacy_allowlist",
    }
    expected = loaders.get(document["serialization"], "data_only")
    if document["load_policy"] != expected:
        raise ContractError("Artifact serialization/load policy mismatch")
    if document["kind"] == "checkpoint" and not document["cpu_loadable"]:
        raise ContractError("Deployable checkpoint must support CPU-safe load")



#### _api

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _api(document):
    datasets = {row["dataset_id"]: row for row in read_json(PROTOCOL)["datasets"]}
    dataset = datasets.get(document["dataset_id"])
    if not dataset or dataset["chapter"] != document["chapter"]:
        raise ContractError("Request dataset/chapter mismatch")
    family = document["family"]
    chapter_families = {2: {"mlp"}, 3: {"cnn"}, 4: {"rnn", "lstm"}, 5: {"kg_rag"}, 6: {"multi_agent"}}
    if family not in chapter_families[document["chapter"]]:
        raise ContractError("Request family/chapter mismatch")
    expected = (
        ["B0", "B1", "B2", "B3", "B4"]
        if family == "kg_rag"
        else ["A0", "A1", "A2"]
        if family == "multi_agent"
        else ["scratch", "keras", "pytorch"]
    )
    if document["implementations"] != expected:
        raise ContractError("Request implementations do not match family")
    kind = document["input"]["kind"]
    if kind != {2: "tabular", 3: "image", 4: "sequence", 5: "question", 6: "question"}[document["chapter"]]:
        raise ContractError("Request payload kind does not match chapter")
    if kind == "sequence" and len({len(row) for row in document["input"]["values"]}) != 1:
        raise ContractError("Sequence rows must have a consistent feature width")



#### _job

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _job(document):
    results = document["results"]
    expected = (
        {"B0", "B1", "B2", "B3", "B4"}
        if document["family"] == "kg_rag"
        else {"A0", "A1", "A2"}
        if document["family"] == "multi_agent"
        else {"scratch", "keras", "pytorch"}
    )
    names = [result["implementation"] for result in results]
    if len(names) != len(set(names)) or any(name not in expected for name in names):
        raise ContractError("Duplicate or inapplicable result slot")
    if document["status"] in {"succeeded", "partial_failed", "failed"} and set(names) != expected:
        raise ContractError("Terminal job must report every implementation, including unavailable slots")
    all_success = len(results) == len(expected) and all(result["status"] == "succeeded" for result in results)
    if any(result["input_sha256"] != document["input_sha256"] for result in results):
        raise ContractError("Job result input hash differs from the shared input hash")
    groups = {result["comparison_group_sha256"] for result in results}
    if document["comparison_valid"] and (len(groups) != 1 or None in groups):
        raise ContractError("Models do not belong to the same comparison group")
    if document["comparison_valid"] and (not all_success or document["comparison_blockers"]):
        raise ContractError("Incomplete or blocked comparison cannot be valid")
    for result in results:
        if result["status"] == "succeeded" and (
            result["prediction"] is None
            or result["error"] is not None
            or result["actual_device"] == "not_run"
            or not result["artifact_sha256"]
        ):
            raise ContractError("Successful result needs prediction, actual device and artifact hash")
        if result["status"] != "succeeded" and (result["error"] is None or result["prediction"] is not None):
            raise ContractError("Failed result needs explicit error, not a fabricated prediction")
        prediction = result["prediction"]
        if (
            prediction
            and prediction["kind"] == "classification"
            and not math.isclose(sum(row["value"] for row in prediction["probabilities"]), 1.0, abs_tol=1e-5)
        ):
            raise ContractError("Classification probabilities must sum to one")



#### _agent

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent(document):
    seen = set()
    agents = set()
    for node in document["nodes"]:
        if node["node_id"] in seen or not set(node["parent_ids"]).issubset(seen):
            raise ContractError("Agent trace must be acyclic and topologically ordered")
        seen.add(node["node_id"])
        agents.add(node["agent"])
    required = {
        "A0": {"direct"},
        "A1": {"router", "specialist"},
        "A2": {"router", "specialist", "verifier", "reporter"},
    }
    if not agents.issubset(required[document["variant"]]):
        raise ContractError("Agent trace uses a role outside its benchmark variant")
    if document["status"] == "succeeded" and not required[document["variant"]].issubset(agents):
        raise ContractError("Agent trace omits required roles")



#### _agent_task

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent_task(document):
    dataset = next(
        (row for row in read_json(PROTOCOL)["datasets"] if row["dataset_id"] == document["dataset_id"]), None
    )
    if not dataset or dataset["chapter"] != 6:
        raise ContractError("Agent task must use a chapter6 task-view dataset")



#### validate_document

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def validate_document(name: str, document: dict, *, publish: bool = False) -> None:
    schema_name, _, definition = name.partition("#")
    schemas = load_schemas()
    if schema_name not in schemas or (definition and definition not in schemas[schema_name].get("$defs", {})):
        raise ContractError(f"Unknown schema: {name}")
    _finite(document)
    registry = Registry().with_resources(
        (schema["$id"], Resource.from_contents(schema)) for schema in schemas.values()
    )
    schema = (
        {"$ref": schemas[schema_name]["$id"] + f"#/$defs/{definition}"}
        if definition
        else schemas[schema_name]
    )
    validator = Draft202012Validator(schema, registry=registry, format_checker=FormatChecker())
    errors = sorted(validator.iter_errors(document), key=lambda error: error.json_path)
    if errors:
        raise ContractError("; ".join(f"{error.json_path}: {error.message}" for error in errors[:5]))
    if publish and document.get("evidence_kind") == "synthetic_fixture":
        raise ContractError("Cannot publish synthetic fixture as measured evidence")
    checks = {
        "dataset": _dataset,
        "split": _split,
        "model": _model,
        "run": _run,
        "seal": _seal,
        "artifact": _artifact,
        "api": _api,
        "api#job": _job,
        "api#agent_task": _agent_task,
        "agent": _agent,
    }
    if name in checks:
        checks[name](document)



#### check_seal_transition

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def check_seal_transition(seal: dict, target: str, *, config_sha256: str) -> None:
    """Validate a proposed transition; do not change files or authorize TEST execution."""
    validate_document("seal", seal)
    if seal["state"] in {"evaluated", "failed_sealed"}:
        raise ContractError("TEST is sealed; do not reopen it")
    if config_sha256 != seal["locked_config_sha256"]:
        raise ContractError("TEST config hash changed")
    allowed = {"draft": {"locked"}, "locked": {"evaluating"}, "evaluating": {"evaluated", "failed_sealed"}}
    if target not in allowed.get(seal["state"], set()):
        raise ContractError("Invalid TEST seal transition")



#### _refs

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _refs(value):
    if isinstance(value, dict):
        if "$ref" in value:
            yield value["$ref"]
        for child in value.values():
            yield from _refs(child)
    elif isinstance(value, list):
        for child in value:
            yield from _refs(child)



#### audit_specification

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def audit_specification(root: Path) -> dict:
    """Audit frozen design documents; never read dataset contents or execute training."""
    errors = []
    result = {"errors": errors, "requirements": 0, "datasets": 0}
    required = [
        "docs/srs_v3.md",
        "docs/traceability_v3.csv",
        "docs/report_style_v3.md",
        "docs/architecture_v3.mmd",
        "docs/risk_register_v3.csv",
        "docs/adr/0001-v3-sources-and-runtime.md",
        "docs/adr/0002-v3-comparison-and-seals.md",
        "docs/adr/0003-v3-api-and-serving.md",
        "docs/adr/0004-v3-formula-minimal-report.md",
        "experiments/configs/v3/protocol_v3.json",
        "report/outline_v3.json",
        "docs/contracts/v3/openapi.json",
    ]
    for relative in required:
        if not (root / relative).is_file():
            errors.append(f"Missing required specification: {relative}")
    if errors:
        return result
    with (root / "docs/traceability_v3.csv").open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    result["requirements"] = len(rows)
    if {row["requirement_id"] for row in rows} != REQUIREMENT_IDS or len(rows) != len(REQUIREMENT_IDS):
        errors.append("Requirement IDs are missing, duplicated or unknown")
    for row in rows:
        if not all(
            row.get(key)
            for key in (
                "origin",
                "requirement",
                "phases",
                "spec_artifact",
                "planned_artifact",
                "acceptance_test",
                "implementation_status",
            )
        ):
            errors.append(f"Incomplete traceability: {row.get('requirement_id')}")
        if not (root / row["spec_artifact"]).is_file():
            errors.append(f"Missing specification artifact for {row['requirement_id']}")
        if any(not token.isdigit() or not 20 <= int(token) <= 35 for token in row["phases"].split(";")):
            errors.append(f"Invalid phase mapping for {row['requirement_id']}")
    protocol = read_json(root / "experiments/configs/v3/protocol_v3.json")
    outline = read_json(root / "report/outline_v3.json")
    datasets = protocol["datasets"]
    result["datasets"] = len(datasets)
    if len(datasets) != 18 or len({row["dataset_id"] for row in datasets}) != 18:
        errors.append("Expected 18 unique dataset/benchmark/view IDs")
    if [row["chapter"] for row in outline["chapters"]] != list(range(1, 7)):
        errors.append("Outline must contain exactly six chapters")
    for chapter in outline["chapters"]:
        expected = {row["dataset_id"] for row in datasets if row["chapter"] == chapter["chapter"]}
        if len(expected) != 3 or set(chapter["dataset_ids"]) != expected:
            errors.append(f"Dataset/outline mismatch: chapter {chapter['chapter']}")
        deploy = [section for section in chapter["sections"] if section["kind"] == "deployment"]
        if chapter["chapter"] >= 2 and (len(deploy) != 1 or deploy[0] != chapter["sections"][-1]):
            errors.append("Deployment must be the last short section, never its own chapter")
    if protocol["ch5"]["new_test_evaluation_allowed"] or protocol["ch5"]["test_run_count"] != 1:
        errors.append("Ch5 TEST must remain sealed at run_count=1")
    if protocol["comparison"]["training_runs_required"] != 4 * 3 * 3 * len(protocol["seeds"]["stability"]):
        errors.append("Neural run matrix count mismatch")
    api_path = root / "docs/contracts/v3/openapi.json"
    api = read_json(api_path)
    if api.get("openapi") != "3.1.1":
        errors.append("OpenAPI version must be explicit")
    for reference in _refs(api):
        relative, _, fragment = reference.partition("#")
        target = (api_path.parent / relative).resolve() if relative else api_path.resolve()
        try:
            target.relative_to(root.resolve())
            content = read_json(target)
            for token in filter(None, fragment.split("/")):
                content = content[token.replace("~1", "/").replace("~0", "~")]
        except (ValueError, KeyError, OSError) as error:
            errors.append(f"Unresolved local API reference {reference}: {error}")
    return result



### `src.common.artifact_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
"""Data-only, hash-verified checkpoints and complete provenance (no pickle)."""

from __future__ import annotations

import hashlib

import io

import json

import re

import zipfile

from pathlib import Path

import numpy as np

from src.common.contracts import canonical_sha256



#### sha256_file

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



#### write_once

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def write_once(path, document):
    path = Path(path)
    encoded = (
        json.dumps(document, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"
    ).encode()
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("xb") as stream:
            stream.write(encoded)
    except FileExistsError:
        if path.read_bytes() != encoded:
            raise ValueError(f"Immutable artifact differs: {path.name}") from None
    return {"path": path, "sha256": hashlib.sha256(encoded).hexdigest()}



#### ArtifactStore

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
class ArtifactStore:
    PROVENANCE_KEYS = (
        "dataset_sha256",
        "split_sha256",
        "subset_sha256",
        "preprocessing_sha256",
        "adapter_sha256",
        "config_sha256",
        "code_sha256",
        "environment_sha256",
        "model_contract_sha256",
    )

    def __init__(self, root, max_bytes=128 * 1024**2):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = max_bytes

    def _path(self, name, suffix):
        if not re.fullmatch(r"[a-z][a-z0-9_-]{0,95}", name):
            raise ValueError("Invalid checkpoint ID")
        path = (self.root / (name + suffix)).resolve()
        if path.parent != self.root:
            raise ValueError("Checkpoint path escapes root")
        return path

    def _provenance(self, provenance):
        if set(provenance) != set(self.PROVENANCE_KEYS) or any(
            not isinstance(v, str) or not re.fullmatch(r"[a-f0-9]{64}", v) for v in provenance.values()
        ):
            raise ValueError("Missing/invalid artifact provenance")

    def save_checkpoint(self, name, arrays, provenance):
        self._provenance(provenance)
        path = self._path(name, ".npz")
        metadata_path = self._path(name, ".json")
        if path.exists() or metadata_path.exists():
            raise ValueError("Checkpoint already exists")
        if not arrays or any(
            not re.fullmatch(r"[a-z][a-z0-9_/-]{0,127}", k)
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Only finite named numeric arrays may be saved")
        if sum(v.nbytes for v in arrays.values()) > self.max_bytes:
            raise ValueError("Checkpoint size budget exceeded")
        buffer = io.BytesIO()
        np.savez(buffer, **arrays)
        payload = buffer.getvalue()
        if len(payload) > self.max_bytes:
            raise ValueError("Serialized size budget exceeded")
        metadata = {
            "checkpoint_version": "phase25.1",
            "serialization": "npz",
            "load_policy": "numpy_no_pickle",
            "cpu_loadable": True,
            "sha256": hashlib.sha256(payload).hexdigest(),
            "size_bytes": len(payload),
            "provenance": dict(provenance),
            "arrays": {k: {"shape": list(v.shape), "dtype": str(v.dtype)} for k, v in arrays.items()},
        }
        with path.open("xb") as stream:
            stream.write(payload)
        write_once(metadata_path, metadata)
        return metadata

    def load_checkpoint(self, name, expected_provenance, *, expected_manifest_sha256=None):
        self._provenance(expected_provenance)
        if (
            expected_manifest_sha256 is not None
            and sha256_file(self._path(name, ".json")) != expected_manifest_sha256
        ):
            raise ValueError("Checkpoint manifest hash mismatch")
        metadata = json.loads(self._path(name, ".json").read_text(encoding="utf-8"))
        path = self._path(name, ".npz")
        if (
            metadata["provenance"] != expected_provenance
            or metadata.get("checkpoint_version") != "phase25.1"
            or metadata.get("load_policy") != "numpy_no_pickle"
        ):
            raise ValueError("Checkpoint provenance/load policy mismatch")
        if (
            path.stat().st_size != metadata["size_bytes"]
            or path.stat().st_size > self.max_bytes
            or sha256_file(path) != metadata["sha256"]
        ):
            raise ValueError("Checkpoint integrity mismatch")
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != metadata["sha256"]:
            raise ValueError("Checkpoint changed while loading")
        with zipfile.ZipFile(io.BytesIO(payload)) as packed:
            if sum(item.file_size for item in packed.infolist()) > self.max_bytes:
                raise ValueError("Expanded checkpoint exceeds size budget")
        with np.load(io.BytesIO(payload), allow_pickle=False) as archive:
            arrays = {k: archive[k] for k in archive.files}
        if set(arrays) != set(metadata["arrays"]) or any(
            list(v.shape) != metadata["arrays"][k]["shape"]
            or str(v.dtype) != metadata["arrays"][k]["dtype"]
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Checkpoint tensor metadata mismatch")
        return arrays



#### code_fingerprint

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def code_fingerprint(root, paths):
    root = Path(root)
    return canonical_sha256({str(path): sha256_file(root / path) for path in sorted(paths)})



### `src.ch05_knowledge_rag.preservation`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
"""Read-only inventory and exact content guards for the migrated Chapter5 baseline."""

import csv

import gzip

import re

from pathlib import PurePosixPath

from src.common.artifact_core import sha256_file, write_once

from src.common.contracts import ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase30"

RUNTIME = "apps/api/models/kg_manifest.json"

RUNTIME_SHA = "8b58e70d6d892cd697e8174ccdbaad78bcd0098a9cd2bc23ca2fda5724823073"

SEAL = "artifacts/manifests/ch5/phase15_test_run.json"

SEAL_SHA = "64f4167eb039903d9a70f56cf6c24a761ec59ad6986faa03f866b97856c7b929"

LOCKED = "experiments/configs/ch5/phase15_locked.json"

LOCKED_SHA = "560c671a3920bd6bde9f7fc005f3c952f857c55969c989bb6109932623210c9e"



#### checked

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def checked(record, root=ROOT):
    name, digest = record["path"], record["sha256"]
    if (
        not isinstance(name, str)
        or not name
        or "\\" in name
        or ":" in name
        or PurePosixPath(name).is_absolute()
        or ".." in PurePosixPath(name).parts
        or not isinstance(digest, str)
        or not re.fullmatch("[a-f0-9]{64}", digest)
    ):
        raise ValueError("Unsafe artifact reference")
    path = (root / name).resolve()
    if not path.is_relative_to(root.resolve()) or not path.is_file():
        raise ValueError("Artifact missing or outside workspace")
    if sha256_file(path) != digest:
        raise ValueError("Artifact hash mismatch: " + name)
    return path



#### ref

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def ref(path, root=ROOT):
    return {"path": path.relative_to(root).as_posix(), "sha256": sha256_file(path)}



#### verify_records

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def verify_records(records, root=ROOT):
    seen = set()
    for row in records:
        if row["path"] in seen:
            raise ValueError("Duplicate artifact reference")
        checked(row, root)
        seen.add(row["path"])
    return len(seen)



#### historical_baseline

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def historical_baseline():
    counts = {"raw": 0, "config": 0, "metric": 0}
    mappings = (
        ("raw", "term_paper/knowledge_rag/data/raw/", "data/raw/ch5/"),
        ("config", "term_paper/knowledge_rag/config/", "experiments/configs/ch5/"),
        ("metric", "term_paper/knowledge_rag/artifacts/metrics/", "artifacts/metrics/ch5/"),
    )
    with gzip.open(
        ROOT / "artifacts/manifests/phase20/before-files.csv.gz", "rt", encoding="utf-8-sig", newline=""
    ) as stream:
        for row in csv.DictReader(stream):
            for kind, old, new in mappings:
                if row["path"].startswith(old):
                    checked({"path": new + row["path"][len(old) :], "sha256": row["sha256"]})
                    counts[kind] += 1
    if counts != {"raw": 11345, "config": 7, "metric": 23}:
        raise ValueError("Historical baseline inventory incomplete")
    checked({"path": SEAL, "sha256": SEAL_SHA})
    checked({"path": LOCKED, "sha256": LOCKED_SHA})
    seal = read_json(ROOT / SEAL)
    if seal["status"] != "COMPLETE" or seal["run_count"] != 1 or seal["locked_config_sha256"] != LOCKED_SHA:
        raise ValueError("Historical TEST seal changed")
    for record in seal["prediction_artifacts"]:
        checked(
            {
                "path": record["path"].replace(
                    "term_paper/knowledge_rag/artifacts/predictions/", "artifacts/predictions/ch5/"
                ),
                "sha256": record["sha256"],
            }
        )
    return {**counts, "sealed_predictions": len(seal["prediction_artifacts"]), "test_run_count": 1}



#### create_snapshot

In [ ]:
%%inline_module src.ch05_knowledge_rag.preservation src/ch05_knowledge_rag/preservation.py both
def create_snapshot():
    path = BASE / "preserved_snapshot.json"
    if path.exists():
        document = read_json(path)
        verify_records(document["files"])
        return document
    print("Phase30: checking original raw/config/metric/TEST baseline", flush=True)
    baseline = historical_baseline()
    print("Phase30: baseline PASS; inventory of preserved migrated assets", flush=True)
    folders = (
        "data/raw/ch5",
        "experiments/configs/ch5",
        "artifacts/manifests/ch5",
        "artifacts/metrics/ch5",
        "artifacts/predictions/ch5",
        "artifacts/figures/ch5",
        "src/term_paper/tools",
        "report/assets/ch5_knowledge_rag",
        "report/references/dataset_cards/ch5",
    )
    files = {
        p
        for folder in folders
        for p in (ROOT / folder).rglob("*")
        if p.is_file() and (folder == "data/raw/ch5" or "__pycache__" not in p.parts)
    }
    files.update((ROOT / "apps/api/models").glob("kg*"))
    files.update((ROOT / "report/references/dataset_cards").glob("ch5*.md"))
    for name in (
        "term_paper/__init__.py",
        "apps/api/knowledge_rag_service.py",
        "report/sections/05_knowledge_rag.md",
        "report/references/bibliography.bib",
        "uv.lock",
        "pyproject.toml",
        "artifacts/manifests/environment.json",
        "artifacts/manifests/phase29/experiment_lock.json",
        "releases/v2/Tieu_luan_AI_ML_CNN_RNN_KG_RAG.docx",
    ):
        files.add(ROOT / name)
    dependencies = []
    for name in ("phase22/spec_lock.json", "phase24/data_lock.json", "phase25/core_lock.json"):
        lock_path = ROOT / "artifacts/manifests" / name
        lock = read_json(lock_path)
        verify_records(lock["files"])
        dependencies.append({**ref(lock_path), "verified_files": len(lock["files"])})
        files.add(lock_path)
    document = {
        "phase": 30,
        "status": "PASS",
        "baseline": baseline,
        "dependencies": dependencies,
        "files": [ref(p) for p in sorted(files)],
        "new_test_evaluation_executed": False,
        "policy": "Read-only original bytes; no rebuild, tuning, public TEST rerun or reserialization",
    }
    write_once(path, document)
    return document



### `src.ch05_knowledge_rag.runtime`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
"""Trusted six-file CPU bundle, lexical graph anchors and frozen rank fusion."""

import gzip

import json

import threading

from pathlib import PurePosixPath

import joblib

from apps.api.knowledge_rag_service import RuntimeBundle

from src.ch05_knowledge_rag.preservation import ROOT, RUNTIME, RUNTIME_SHA, checked

from src.common.contracts import read_json

from term_paper.tools.phase13_graph import normalize_label

from term_paper.tools.phase14_rag import reciprocal_rank_fusion

from term_paper.tools.phase15_kgrag import graph_filter_ranking, weighted_rrf



#### validate_layout

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
def validate_layout(manifest):
    files = [row["path"] for row in manifest["files"]]
    roles = [manifest["chunks_file"], manifest["graph_file"]]
    if set(manifest["indexes"]) != {"medical", "finance"}:
        raise ValueError("Require exact two retrieval domains")
    for domain in ("medical", "finance"):
        if set(manifest["indexes"][domain]) != {"sparse", "dense"}:
            raise ValueError("Require sparse/dense index roles")
        roles.extend(manifest["indexes"][domain].values())
    if len(files) != 6 or len(set(files)) != 6 or len(set(roles)) != 6 or set(files) != set(roles):
        raise ValueError("Exact six verified artifacts and distinct role references required")
    if any(not name or len(PurePosixPath(name).parts) != 1 or ":" in name or "\\" in name for name in files):
        raise ValueError("Unsafe runtime filename")
    return set(files)



#### graph_candidates

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
def graph_candidates(question, graph):
    normalized = normalize_label(question)
    terms = set(normalized.split())
    matched = [
        (key, row)
        for key, row in graph.items()
        if key and (key in normalized or (len(key.split()) > 1 and set(key.split()) <= terms))
    ]
    matched.sort(key=lambda pair: (-len(pair[0]), pair[0]))
    paths = {}
    for _, row in matched[:4]:
        for chunk_id in row["chunk_ids"]:
            paths.setdefault(chunk_id, [row["node_id"], chunk_id])
    return set(paths), paths



#### compare_rankings

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
def compare_rankings(sparse, dense, candidates, sparse_weight, weights, top_k, rrf_k):
    graph = graph_filter_ranking(sparse, candidates, 100)
    return {
        "B0": sparse[:top_k],
        "B1": dense[:top_k],
        "B2": reciprocal_rank_fusion(sparse, dense, sparse_weight, top_k, rrf_k),
        "B3": graph[:top_k],
        "B4": weighted_rrf({"sparse": sparse, "dense": dense, "graph": graph}, weights, top_k, rrf_k),
    }



#### _read_gzip

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
def _read_gzip(path, max_bytes):
    with gzip.open(path, "rb") as stream:
        payload = stream.read(max_bytes + 1)
    if len(payload) > max_bytes:
        raise ValueError("Runtime decompressed budget exceeded")
    return payload.decode("utf-8")



#### load_bundle

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
def load_bundle(root=ROOT):
    manifest_path = checked({"path": RUNTIME, "sha256": RUNTIME_SHA}, root)
    manifest = read_json(manifest_path)
    validate_layout(manifest)
    total = 0
    for record in manifest["files"]:
        name = "apps/api/models/" + record["path"]
        path = checked({"path": name, "sha256": record["sha256"]}, root)
        if path.stat().st_size != record["size_bytes"]:
            raise ValueError("Runtime artifact size mismatch")
        total += path.stat().st_size
    if total != manifest["package_bytes"] or total > min(manifest["artifact_budget_bytes"], 192 * 1024**2):
        raise ValueError("Runtime package budget mismatch")
    folder = manifest_path.parent
    chunks = {"medical": {}, "finance": {}}
    for line in _read_gzip(folder / manifest["chunks_file"], 256 * 1024**2).splitlines():
        row = json.loads(line)
        domain, key = row["domain"], row["chunk_id"]
        if domain not in chunks or key in chunks[domain]:
            raise ValueError("Duplicate or unknown-domain chunk")
        chunks[domain][key] = row
    if {name: len(rows) for name, rows in chunks.items()} != manifest["counts"]["chunks"]:
        raise ValueError("Runtime chunk count mismatch")
    graph = json.loads(_read_gzip(folder / manifest["graph_file"], 64 * 1024**2))
    for domain, rows in chunks.items():
        if len(graph[domain]) != manifest["counts"]["graph_anchors"][domain]:
            raise ValueError("Runtime graph count mismatch")
        valid_ids = set(rows)
        if any(not set(row["chunk_ids"]) <= valid_ids for row in graph[domain].values()):
            raise ValueError("Dangling compact graph reference")
    # Only the hard-pinned manifest's four local joblib files reach deserialization.
    # Never accept uploaded pickles, user paths, or a replacement manifest.
    indexes = {
        domain: {kind: joblib.load(folder / name) for kind, name in roles.items()}
        for domain, roles in manifest["indexes"].items()
    }
    for domain, roles in indexes.items():
        if any(set(index.doc_ids) != set(chunks[domain]) for index in roles.values()):
            raise ValueError("Index/corpus ID mismatch")
    return RuntimeBundle(manifest, chunks, indexes, graph)



#### LazyBundle

In [ ]:
%%inline_module src.ch05_knowledge_rag.runtime src/ch05_knowledge_rag/runtime.py both
class LazyBundle:
    def __init__(self, loader=load_bundle):
        self.loader = loader
        self.value = None
        self.lock = threading.Lock()

    def get(self):
        with self.lock:
            if self.value is None:
                self.value = self.loader()
        return self.value



### `src.ch05_knowledge_rag.service`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch05_knowledge_rag.service src/ch05_knowledge_rag/service.py both
"""V3 local request/job contract. No evaluator, fitting, network or arbitrary tools."""

import time

from datetime import UTC, datetime, timedelta

from apps.api.knowledge_rag_service import guardrail_reason, validate_question

from src.ch05_knowledge_rag.preservation import RUNTIME_SHA, checked

from src.ch05_knowledge_rag.runtime import LazyBundle, compare_rankings, graph_candidates, load_bundle

from src.common.contracts import canonical_sha256, read_json, validate_document

from term_paper.tools.phase14_rag import grounded_answer

DATASETS = ("ch5_medquad_diabetes", "ch5_finqa", "ch5_wikidata_subgraphs")

METHODS = ("B0", "B1", "B2", "B3", "B4")

DOMAINS = {DATASETS[0]: "medical", DATASETS[1]: "finance"}

SOURCES = {
    DATASETS[0]: "https://github.com/abachaa/MedQuAD/tree/577bd37b96c02d1833b2c9eed2de9f96964e96cb",
    DATASETS[1]: "https://github.com/czyssrs/FinQA/tree/0f16e2867befa6840783e58be38c9efb9229d742",
}

WARNING = "Academic retrospective demo, not medical or investment advice. Citations identify the preserved runtime chunk container, not verified original per-chunk URLs."

SCOPE = "phase30.compact_free_question_demo.1"



#### sample_request

In [ ]:
%%inline_module src.ch05_knowledge_rag.service src/ch05_knowledge_rag/service.py both
def sample_request(dataset):
    if dataset not in DATASETS:
        raise ValueError("Unknown Chapter5 dataset")
    question = {
        DATASETS[0]: "Diabetes complications include kidney disease: find supporting evidence.",
        DATASETS[1]: "what is the the interest expense in 2009?",
        DATASETS[2]: "Q12206",
    }[dataset]
    return {
        "schema_version": "3.0.0",
        "chapter": 5,
        "dataset_id": dataset,
        "family": "kg_rag",
        "implementations": list(METHODS),
        "input": {"kind": "question", "text": question, "domain": DOMAINS.get(dataset, "auto")},
    }



#### validate_request

In [ ]:
%%inline_module src.ch05_knowledge_rag.service src/ch05_knowledge_rag/service.py both
def validate_request(request):
    validate_document("api", request)
    if request["chapter"] != 5 or request["dataset_id"] not in DATASETS or request["family"] != "kg_rag":
        raise ValueError("Require Chapter5 KG–RAG request")
    expected = DOMAINS.get(request["dataset_id"])
    if expected and request["input"]["domain"] not in {expected, "auto"}:
        raise ValueError("Dataset/domain mismatch")
    if not expected and request["input"]["domain"] != "auto":
        raise ValueError("Wikidata structural view requires domain=auto")
    return validate_question(request["input"]["text"])



#### KgRagService

In [ ]:
%%inline_module src.ch05_knowledge_rag.service src/ch05_knowledge_rag/service.py both
class KgRagService:
    def __init__(self, loader=load_bundle, artifact_sha256=RUNTIME_SHA, chunks_sha256=None):
        self.runtime = LazyBundle(loader)
        self.artifact_sha256 = artifact_sha256
        self.chunks_sha256 = (
            chunks_sha256 or "42f025f955852567436dc7cd7efde701ca014429f47dacae7d7c0ba23ab51f29"
        )
        self.sparse_weight = 0.75

    def compare(self, request):
        question = validate_request(request)
        dataset = request["dataset_id"]
        domain = DOMAINS.get(dataset)
        input_hash = canonical_sha256(request["input"])
        group = canonical_sha256({"scope": SCOPE, "dataset": dataset, "artifact": self.artifact_sha256})
        suffix = canonical_sha256(request)[:16]
        now = datetime.now(UTC)
        structural = domain is None
        reason = guardrail_reason(question, domain)
        results, evidence, paths = [], {}, {}
        rankings, bundle, timing = {}, None, {method: 0.0 for method in METHODS}
        unavailable = False
        if not structural and not reason:
            try:
                config_path = checked(
                    {
                        "path": "experiments/configs/ch5/phase14_locked.json",
                        "sha256": "ea5b3546a8e408648f5024d62bd0cb3584784ead2f594e126d2dea356240653d",
                    }
                )
                self.sparse_weight = read_json(config_path)["selected_sparse_weight"]
                bundle = self.runtime.get()
            except (OSError, ValueError, KeyError, ImportError, RuntimeError):
                unavailable = True
        if bundle is not None:
            config = bundle.manifest["retrieval"]
            start = time.perf_counter()
            sparse = bundle.indexes[domain]["sparse"].search(question, config["candidate_depth"])
            sparse_ms = (time.perf_counter() - start) * 1000
            start = time.perf_counter()
            dense = bundle.indexes[domain]["dense"].search(question, config["candidate_depth"])
            dense_ms = (time.perf_counter() - start) * 1000
            start = time.perf_counter()
            candidates, paths = graph_candidates(question, bundle.graph[domain])
            graph_ms = (time.perf_counter() - start) * 1000
            start = time.perf_counter()
            rankings = compare_rankings(
                sparse,
                dense,
                candidates,
                self.sparse_weight,
                config["weights"],
                config["top_k"],
                config["rrf_k"],
            )
            fusion_ms = (time.perf_counter() - start) * 1000
            timing = {
                "B0": sparse_ms,
                "B1": dense_ms,
                "B2": sparse_ms + dense_ms + fusion_ms,
                "B3": sparse_ms + graph_ms,
                "B4": sparse_ms + dense_ms + graph_ms + fusion_ms,
            }
        for method in METHODS:
            row = {
                "implementation": method,
                "status": "succeeded",
                "model_id": f"{dataset}_{method.lower()}_compact_demo",
                "actual_device": "cpu",
                "actual_backend": "numpy",
                "inference_ms": timing[method],
                "artifact_sha256": self.artifact_sha256,
                "input_sha256": input_hash,
                "comparison_group_sha256": group,
                "error": None,
            }
            if structural or unavailable:
                row.update(
                    status="unavailable",
                    actual_device="not_run",
                    actual_backend="not_run",
                    inference_ms=None,
                    artifact_sha256=None,
                    prediction=None,
                    comparison_group_sha256=None,
                    error={
                        "code": "artifact_unavailable" if unavailable else "structural_dataset_only",
                        "message": "Verified runtime unavailable."
                        if unavailable
                        else "Wikidata is a snapshot browser, not a third B0–B4 QA benchmark.",
                    },
                )
                results.append(row)
                continue
            rank = rankings.get(method, [])
            evidence[method] = []
            if reason:
                answer = {
                    "answer": "Refused: " + reason,
                    "citations": [],
                    "abstained": True,
                    "reason": reason,
                }
            else:
                start = time.perf_counter()
                chunks = bundle.chunks[domain]
                answer_config = bundle.manifest["answer_layer"]
                answer = grounded_answer(
                    question,
                    rank,
                    {key: chunks[key]["text"] for key, _ in rank},
                    answer_config["minimum_overlap_terms"],
                    answer_config["minimum_overlap_ratio"],
                    answer_config["unsafe_patterns"],
                    answer_config["maximum_characters"],
                )
                row["inference_ms"] += (time.perf_counter() - start) * 1000
                evidence[method] = [
                    {
                        "chunk_id": key,
                        "document_id": chunks[key]["document_id"],
                        "text": chunks[key]["text"],
                        "score": float(score),
                    }
                    for key, score in rank
                ]
            citations = [
                {
                    "evidence_id": "chunk_" + canonical_sha256(key)[:24],
                    "source_url": SOURCES[dataset],
                    "locator": f"kg_chunks.jsonl.gz#{key};document={bundle.chunks[domain][key]['document_id']}",
                    "source_sha256": self.chunks_sha256,
                }
                for key in answer["citations"]
            ]
            row["prediction"] = {
                "kind": "answer",
                "text": answer["answer"] or "Insufficient evidence.",
                "abstained": answer["abstained"],
                "citations": citations,
            }
            results.append(row)
        job = {
            "schema_version": "3.0.0",
            "request_id": "kg_request_" + suffix,
            "job_id": "kg_demo_" + suffix,
            "request_sha256": canonical_sha256(request),
            "status": "failed" if structural or unavailable else "succeeded",
            "dataset_id": dataset,
            "family": "kg_rag",
            "created_at": now.isoformat(),
            "expires_at": (now + timedelta(hours=1)).isoformat(),
            "results": results,
            "comparison_valid": not structural and not reason and not unavailable,
            "comparison_blockers": ["Structural view only"]
            if structural
            else (
                ["Verified runtime unavailable"]
                if unavailable
                else (["Safety refusal; no ranker executed"] if reason else [])
            ),
            "trace_ids": [],
            "metric_artifacts": [],
            "input_sha256": input_hash,
        }
        validate_document("api#job", job)
        retrieved = {row["chunk_id"] for rows in evidence.values() for row in rows}
        paths = {key: path for key, path in paths.items() if key in retrieved}
        return {
            "job": job,
            "evidence": evidence,
            "graph_paths": paths,
            "warning": WARNING,
            "scope": SCOPE,
            "refusal_reason": reason,
            "public_test_evaluation_executed": False,
            "timing_scope": "Shared retrieval components + answer CPU milliseconds; excludes cold load and serialization; not independent per-method benchmark",
        }



### `src.ch05_knowledge_rag.structural`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch05_knowledge_rag.structural src/ch05_knowledge_rag/structural.py both
"""Offline entity/revision browser with exact trusted snapshot provenance."""

import re

from src.ch05_knowledge_rag.preservation import checked

from src.common.contracts import read_json

INDEX = "data/raw/ch5/wikidata/snapshot_index.json"

INDEX_SHA = "1ba148c6369a68963b0da049a58fea722372ba213f57e0d88e402f3d587ac64e"



#### inspect_entity

In [ ]:
%%inline_module src.ch05_knowledge_rag.structural src/ch05_knowledge_rag/structural.py both
def inspect_entity(qid):
    if not isinstance(qid, str) or not re.fullmatch(r"Q[1-9][0-9]{0,12}", qid):
        raise ValueError("Require an allowlisted Wikidata QID, not a path or URL")
    index = read_json(checked({"path": INDEX, "sha256": INDEX_SHA}))
    entry = next((r for r in index["entities"] if r["qid"] == qid), None)
    if entry is None:
        raise ValueError("QID not present in the frozen ten-entity snapshot")
    path = checked(
        {"path": "data/raw/ch5/" + entry["entity_relative_path"], "sha256": entry["entity_sha256"]}
    )
    entity = read_json(path)["entities"][qid]
    count = sum(len(items) for items in entity["claims"].values())
    if entity["lastrevid"] != entry["revision_id"] or count != entry["claim_count"]:
        raise ValueError("Entity revision/count mismatch")
    return {
        "qid": qid,
        "label": entity.get("labels", {}).get("en", {}).get("value", qid),
        "description": entity.get("descriptions", {}).get("en", {}).get("value", ""),
        "revision_id": entity["lastrevid"],
        "claim_count": count,
        "property_count": len(entity["claims"]),
        "sample_claims": [
            {"property": key, "statement": entity["claims"][key][0]} for key in sorted(entity["claims"])[:5]
        ],
        "citation": {
            "evidence_id": "wikidata_" + qid.lower(),
            "source_url": entry["entity_endpoint"],
            "locator": f"{qid}@{entry['revision_id']}",
            "source_sha256": entry["entity_sha256"],
        },
        "qa_benchmark": False,
        "training_executed": False,
        "test_evaluation_executed": False,
    }



## Thực hành: ch5_finqa

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# KG–RAG / ch5_finqa: B0–B4 CPU

Demo dùng snapshot/index đã khóa, không fit hoặc mở TEST. Medical là câu minh họa tự soạn không gán split; FinQA là câu TRAIN từ EDA card. Wikidata chỉ duyệt entity/revision/hash. Chạy trong kernel WSL V3; không cần API key. Graph compact/free-question không đồng nhất evaluator TEST lịch sử; timing không phải benchmark độc lập. Không tư vấn y khoa/tài chính.

In [ ]:
if RUN_EXPERIMENTS:
    from pathlib import Path
    import sys
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists())
    if str(root) not in sys.path: sys.path.insert(0, str(root))
    from src.ch05_knowledge_rag.service import KgRagService, sample_request
    service = globals().get('phase30_service') or KgRagService()
    request = sample_request('ch5_finqa')
    result = service.compare(request)
    assert result['job']['status'] == 'succeeded'
    assert result['job']['comparison_valid']
    assert len(result['job']['results']) == 5
    for row in result['job']['results']:
        print(row['implementation'], row['actual_backend'], row['actual_device'], row['prediction']['abstained'], row['prediction']['text'], row['prediction']['citations'])
    assert not result['public_test_evaluation_executed']
    assert not any(name in sys.modules for name in ('torch','tensorflow','keras'))
    print('No TEST evaluation, fit, download or external API call executed.')


B0 numpy cpu False millions of dollars the interest expense of 2009 is -600 ( 600 ) ; the interest expense of 2008 is -511 ( 511 ) ; the interest expense of 2007 is -482 ( 482 ) ; the interest expense of % ( % ) change 2009 v 2008 is 17 ; the interest expense of % ( % ) change 2008 v 2007 is 6 ; [{'evidence_id': 'chunk_a63e6c5181bbb843e9c4688e', 'source_url': 'https://github.com/czyssrs/FinQA/tree/0f16e2867befa6840783e58be38c9efb9229d742', 'locator': 'kg_chunks.jsonl.gz#fin-chunk-ed0c43d76614b7ac8c13;document=fin-doc-6e6f9ce24302aa2903b6', 'source_sha256': '42f025f955852567436dc7cd7efde701ca014429f47dacae7d7c0ba23ab51f29'}]
B1 numpy cpu False interest expense . [{'evidence_id': 'chunk_698f0e7ee2c6288a1120e300', 'source_url': 'https://github.com/czyssrs/FinQA/tree/0f16e2867befa6840783e58be38c9efb9229d742', 'locator': 'kg_chunks.jsonl.gz#fin-chunk-05f557361819f0105a2e;document=fin-doc-89ff28551f3c25be5f64', 'source_sha256': '42f025f955852567436dc7cd7efde701ca014429f47dacae7d7c0ba23ab51f2

## Thực hành: ch5_medquad_diabetes

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# KG–RAG / ch5_medquad_diabetes: B0–B4 CPU

Demo dùng snapshot/index đã khóa, không fit hoặc mở TEST. Medical là câu minh họa tự soạn không gán split; FinQA là câu TRAIN từ EDA card. Wikidata chỉ duyệt entity/revision/hash. Chạy trong kernel WSL V3; không cần API key. Graph compact/free-question không đồng nhất evaluator TEST lịch sử; timing không phải benchmark độc lập. Không tư vấn y khoa/tài chính.

In [ ]:
if RUN_EXPERIMENTS:
    from pathlib import Path
    import sys
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists())
    if str(root) not in sys.path: sys.path.insert(0, str(root))
    from src.ch05_knowledge_rag.service import KgRagService, sample_request
    service = globals().get('phase30_service') or KgRagService()
    request = sample_request('ch5_medquad_diabetes')
    result = service.compare(request)
    assert result['job']['status'] == 'succeeded'
    assert result['job']['comparison_valid']
    assert len(result['job']['results']) == 5
    for row in result['job']['results']:
        print(row['implementation'], row['actual_backend'], row['actual_device'], row['prediction']['abstained'], row['prediction']['text'], row['prediction']['citations'])
    assert not result['public_test_evaluation_executed']
    assert not any(name in sys.modules for name in ('torch','tensorflow','keras'))
    print('No TEST evaluation, fit, download or external API call executed.')


B0 numpy cpu False is Important Diabetes is a serious disease that can lead to a number of health problems such as heart disease, stroke, vision problems, kidney disease and even death. Sometimes people have symptoms but do not suspect diabetes. They delay scheduling a checkup because they do not feel sick. Many people do not find out they have the disease until they have diabetes complications, such as a heart attack or stroke.… [{'evidence_id': 'chunk_f953b0cd71b15c1de43df150', 'source_url': 'https://github.com/abachaa/MedQuAD/tree/577bd37b96c02d1833b2c9eed2de9f96964e96cb', 'locator': 'kg_chunks.jsonl.gz#med-chunk-e5068975c5ccace8d6b0;document=med-doc-24832e0a12ba7cf022f0', 'source_sha256': '42f025f955852567436dc7cd7efde701ca014429f47dacae7d7c0ba23ab51f29'}]
B1 numpy cpu False is Important Diabetes is a serious disease that can lead to a number of health problems such as heart disease, stroke, vision problems, kidney disease and even death. Sometimes people have symptoms but do not s

## Thực hành: ch5_wikidata_subgraphs

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

**Bản đường dẫn r2.** Notebook gốc và output được giữ trong provenance. Kernel: Python3.11 WSL NumPy/PyTorch. Không Run All pipeline huấn luyện chỉ để xem kết quả đã lưu.


# Snapshot Wikidata: cấu trúc, không phải QA TEST

Demo dùng snapshot/index đã khóa, không fit hoặc mở TEST. Medical là câu minh họa tự soạn không gán split; FinQA là câu TRAIN từ EDA card. Wikidata chỉ duyệt entity/revision/hash. Chạy trong kernel WSL V3; không cần API key. Graph compact/free-question không đồng nhất evaluator TEST lịch sử; timing không phải benchmark độc lập. Không tư vấn y khoa/tài chính.

In [ ]:
if RUN_EXPERIMENTS:
    from pathlib import Path
    import sys
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists())
    if str(root) not in sys.path: sys.path.insert(0, str(root))
    from src.ch05_knowledge_rag.structural import inspect_entity
    from src.common.contracts import read_json
    index = read_json(root / 'data/raw/ch5/wikidata/snapshot_index.json')
    result = [inspect_entity(r['qid']) for r in index['entities']]
    for row in result: print(row['qid'], row['revision_id'], row['claim_count'], row['citation']['source_sha256'])
    assert len(result) == 10 and sum(r['claim_count'] for r in result) == 2056
    assert all(not r['qa_benchmark'] for r in result)
    assert not any(name in sys.modules for name in ('torch','tensorflow','keras'))
    print('No TEST evaluation, fit, download or external API call executed.')


Q12206 2554502707 186 d78017cab4a383f45194004bdcbeaf0aa371b592bfe28119ba26ba3634dd41c1
Q124407 2556005597 125 4142195152d0d29a458a58660970b42f1d14a442b641a474a2cecf663f98b7d9
Q3025883 2549752035 161 96e23996f6da47f947c9e8d37e88359f8f88a2cf0ae8678ae59a3f5c5b581766
Q50265665 2555721146 125 995bb7156953a9fe34b2916206540e0f5e00701a66e8310a8964f27ae8c68470
Q271993 2549698902 62 524f12f865bc799c5441c319b50aadc52795d86cc8511c126d73da804409029d
Q192907 2556173446 70 aef93668311a49e75979dadf9ad5f99eea76470be3ead310a8c700d76c7afc79
Q699735 2551742693 51 e2215c3b8ec342c6cd8e2905518201917c3de76c9859112d40ad07e1f5fa67c3
Q850210 2547192660 39 d21a0b5f9c88ec41ef5a518b71107d27b84b9365ba9c42903c647ab38db452e2
Q570938 2551809219 18 80b313891144ec81921c3c8d12622600edf75b186f79ab50eaa96ae8dd795dec
Q312 2548367127 1219 10cbb21e4cd9091de975482a74f16cde8bb0ae7639cdd8c6f3605915bec4e7a3
No TEST evaluation, fit, download or external API call executed.


In [ ]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})
